# Suite STO — Stores

A store is the root object of a body of data: it holds schemas by name and the objects built with them, and everything
that looks a schema up takes one. `Proxies.OfStore` and `Bindings.OfStore` implement the protocol `Stores.Store`.
Stores are isolated from one another; objects move between them as snapshots.

In [ ]:
import { Bindings as Bd, JSON, Modules, Proxies, Schemas as S, Validators } from "@mbse/schemas/Framework";
import { AttributeError, LookupError, ValueError } from "@mbse/schemas/Framework/Errors";
import { assert, conformance_problems, instance_problems, raises, same, text } from "./support.js";

let Item = new S.OfObject.Builder().ref().properties(text("name")).create();
const Holds = new S.OfRelation.Builder().links("owner", "item").properties(text("label")).create();
const Box = new S.OfObject.Builder().ref().properties(text("name")).relations((r) => r.name("items").of(Holds).me("owner")).create();
Item = new S.OfObject.Builder(Item).relations((r) => r.name("boxes").of(Holds).me("item")).update();
const Port = new S.OfObject.Builder().properties(text("name")).relations((r) => r.name("wired").of(Holds).me("owner")).create();
const Card = new S.OfObject.Builder().ref().properties((p) => p.name("port").of(Port)).create();
const SCHEMAS = new Map<string, S.OfObject.Data | S.OfRelation.Data>([["Item", Item], ["Box", Box], ["Holds", Holds], ["Card", Card]]);

/** A proxy store with the schemas named, of Item, Box, Holds and Card. */
function stocked(...names: string[]): Proxies.OfStore {
  const store = new Proxies.OfStore();
  for (const name of names) store.register(name, SCHEMAS.get(name) as S.OfObject.Data);
  return store;
}

## STO-01 · A store holds schemas by name

In [ ]:
{
  const store = stocked("Item", "Box", "Holds");
  assert(same(store.names(), ["Schemas.Module", "Item", "Box", "Holds"])); // every proxy store starts with the meta-schemas
  assert(store.schema("Box") === Box && store.registered("Holds") === Holds && store.name_of(Item) === "Item");
  raises(ValueError, () => store.register("Box", Box), "schema 'Box' is already registered");
  raises(AttributeError, () => store.schema("Nope"), "no schema registered as 'Nope'");
  raises(TypeError, () => store.schema("Holds"), "'Holds' is a relation; no relation builder is exposed");
  raises(LookupError, () => store.registered("Nope"), "no schema registered as 'Nope'");
  raises(LookupError, () => store.name_of(Port), "schema is not registered");
  store.register("register", Item); // a name a method shadows is reached through builder()
  assert(store.builder("register").name("r").create().schema_name() === "register");
  raises(AttributeError, () => store["_hidden"], "_hidden");
  assert(store.Box().name("b").create().name === "b" && store.builder("Box").name("c").create().name === "c");
}

## STO-02 · A store holds the objects built through it

In [ ]:
{
  const store = stocked("Item", "Box", "Holds", "Card");
  assert(store.extent("Box").length === 0 && store.extent("Card").length === 0);
  const a = store.Item().name("a").create();
  const box = store.Box().name("box").items((e: any) => e.item(a).label("first")).create();
  const box2 = store.Box(box).name("box2").clone();
  assert(store.Box(box).name("renamed").update() === box); // an update makes nothing new
  store.Box(box).items((e: any) => e.item((i: any) => i.name("inline"))).update(); // an object created through a link
  assert(same(store.extent("Box"), [box, box2]));
  assert(same(store.extent("Item").map((i) => i.name), ["a", "inline"]));
  const card = store.Card().port((p: any) => p.name("p1")).create();
  assert(same(store.extent("Card"), [card])); // its value object is reached through it, not an extent
  const loaded = JSON.FromJSON(store).Reachable(Box, JSON.ToJSON(store).Reachable(Box, box)); // decoding builds in the store
  // The snapshot holds box2 too, which shares an item with box; the root is built first.
  assert(same(store.extent("Box").map((b) => b.name), ["renamed", "box2", "renamed", "box2"]) && store.extent("Box")[2] === loaded);
  assert(store.extent("Item").length === 4);
  raises(AttributeError, () => store.extent("Nope"), "no schema registered as 'Nope'");
  raises(TypeError, () => store.extent("Holds"), "'Holds' is a relation");
}

## STO-03 · Stores are isolated; objects move between them as snapshots

In [ ]:
{
  const [one, two] = [stocked("Item", "Box", "Holds"), stocked("Item", "Box", "Holds")];
  const Other = new S.OfObject.Builder().ref().properties(text("title")).create();
  const three = new Proxies.OfStore();
  three.register("Box", Other); // the same name, another schema
  assert(three.Box().title("t").create().title === "t");
  const x = one.Item().name("x").create();
  const y = two.Item().name("y").create();
  raises(TypeError, () => one.Box().items((e: any) => e.item(y)), "the object belongs to another store");
  raises(TypeError, () => two.Item(x), "the object belongs to another store");
  const b = one.Box().name("b").items((e: any) => e.item(x).label("l")).create();
  assert(Proxies.store_of(b) === one && Proxies.store_of(b) !== two);
  const moved = JSON.FromJSON(two).Reachable(Box, JSON.ToJSON(one).Reachable(Box, b)) as Proxies.Instance;
  assert(Proxies.store_of(moved) === two && moved !== b && same(two.extent("Item").map((i) => i.name), ["y", "x"]));
  assert(Validators.Validate(two).Reachable(Box, moved).length === 0);
  assert(same(Validators.Validate(three)(Box, b), ["the value is a 'Box', not an instance of the given schema"]));
  const card = stocked("Card").Card().port((p: any) => p.name("p")).create();
  assert(Proxies.store_of(card.port) === Proxies.store_of(card)); // a value object's store is its owner's
  raises(TypeError, () => Proxies.store_of("text"), "the value does not belong to a store");
  const module = Modules.module(one, { Item }) as Proxies.Instance;
  assert(Proxies.store_of(module) === one && same(one.extent("Schemas.Module"), [module]));
}

## STO-04 · Every implementation meets the protocol

In [ ]:
{
  assert(conformance_problems(Proxies.OfStore, "Store").length === 0);
  assert(conformance_problems(Bd.OfStore, "Store").length === 0);
  assert(instance_problems(new Proxies.OfStore(), "Store").length === 0);

  class Thing {
    constructor(public name: string | null = null) {}

    identity(): unknown {
      return this;
    }

    schema_name(): string {
      return "Item";
    }

    owner(): null {
      return null;
    }

    accept(visitor: any): void {
      Bd.accept(THING, this, visitor);
    }
  }

  const nameOf = (s: Bd.State) => (s.values.get("name") as string | undefined) ?? null;
  const THING = new Bd.Binding(Item, (t: Thing) => new Bd.State(new Map([["name", t.name]])), (s) => new Thing(nameOf(s)),
    (t: Thing, s) => Object.assign(t, { name: nameOf(s) }), { implied: ["boxes"] });
  const bound = new Bd.OfStore(new Map([["Item", [Item, (instance?: Thing) => new Bd.Builder(THING, instance)] as const]]),
    new Map([["Holds", Holds]]));
  assert(instance_problems(bound, "Store").length === 0 && bound.extent("Item").length === 0);
  assert((bound as any).then === undefined); // JavaScript's probes are not schema lookups
  assert(same(bound.names(), ["Item", "Holds"])); // a store of bound classes holds no meta-schemas
  const first = (bound as any).Item().property("name", (p: any) => p.value((a: any) => a.as_native((n: any) => n.set("first")))).create();
  const copy = bound.builder("Item", first).clone();
  assert(bound.builder("Item", first).update() === first);
  new Bd.Builder(THING).create(); // a builder the store did not make adds nothing
  assert(same(bound.extent("Item"), [first, copy]) && bound.member(first, "name") === "first");
  assert((JSON.FromJSON(bound)(Item, JSON.ToJSON(bound)(Item, first)) as Thing).name === "first" && bound.extent("Item").length === 3);
  raises(AttributeError, () => (bound as any).Nope, "Nope");
  raises(TypeError, () => bound.builder("Holds"), "'Holds' is a relation");
  raises(TypeError, () => bound.extent("Holds"), "'Holds' is a relation");
}

## STO-05 · A value object not yet placed belongs to no store

In [ ]:
{
  const P = Proxies._internals;
  const builder = new P.RecordBuilderTarget(Port);
  builder.property("name", (p) => p.value((a) => a.as_native((n) => n.set("loose"))));
  const loose = builder.build(); // built, but not placed in an owner
  const seen: string[] = [];
  loose.accept({ identify: () => undefined, property: (n: string) => seen.push(n), adjacency: (n: string) => seen.push(n) } as any);
  assert(same(seen, ["name"]) && loose.schema_name() === ""); // its properties, and no entries; its schema is named nowhere
  raises(TypeError, () => Proxies.store_of(loose), "the value does not belong to a store");
  raises(TypeError, () => new P.RecordBuilderTarget(Port).adjacency("wired", (a) => a.add((e: any) => e.item((i: any) => i.name("new")))),
    "an object is created through a link only by a store's builder");
}